In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is engaged in a light-hearted and imaginative conversation about a fictional place called Lunapolis, exploring aspects such as its weather, population, and potential labor issues.\n\n## SUMMARY\n\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as having clear skies, with temperatures reaching a high of 120°C and a low of -100°C.\n- Lunapolis has a population of 100,000 cheese miners.\n- There is a suggestion that the cheese miners' union may go on strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\nContinue the conversation to explore additional fictional aspects of Lunapolis or address related questions.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='affe31b5-b10a-4a91-b2ee-2a25764a516d'),
              HumanMessage(content="If you were Lunapolis' 

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is engaged in a light-hearted and imaginative conversation about a fictional place called Lunapolis, exploring aspects such as its weather, population, and potential labor issues.

## SUMMARY

- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is described as having clear skies, with temperatures reaching a high of 120°C and a low of -100°C.
- Lunapolis has a population of 100,000 cheese miners.
- There is a suggestion that the cheese miners' union may go on strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

Continue the conversation to explore additional fictional aspects of Lunapolis or address related questions.


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='3c9ba7d5-4809-4999-98f7-e26eb118cc7d'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='19fa626d-6e04-4dff-b252-b6721093a5b0', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='2cc9125c-af1e-4244-ae01-1abf55e12feb'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='69b91f81-16a7-49a7-8370-d83efee6c191', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='0f8bde44-c496-46f3-852f-8e453cfbd9d6'),
              AIMessage(content='I can’t read your device’s temperature from here. If the devic

In [8]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If the device isn’t powering on, you won’t be able to measure temps until it’s on again. If you want, tell me your device type (laptop/desktop/phone/tablet), the model, and the OS, and I’ll give you exact steps to check temps.

General guidance you can try now:
- If it feels very hot, turn it off and unplug it. Let it cool in a well-ventilated area for at least 15–30 minutes.
- Check that vents aren’t blocked by dust or a case; if possible, clean the vents with compressed air.
- If it has a removable battery, try powering with the charger plugged in with the battery out (where safe) or with a known-good charger.
- After cooling, try a safe reboot. If it still won’t turn on, there may be a hardware issue beyond temperature.

If you want monitoring steps once the device can power on:
- Windows: use HWMonitor or Open Hardware Monitor to view CPU/GPU temps.
- macOS: apps like iStat Menus or Macs Fan Control (or check temps with some built-i